# Notebook 2: knowledge distillation, teacher -> YOLOv8-seg (train, evaluate, save)

This notebook builds the **segmentation model that runs on the Raspberry Pi**:

```
folder tree of videos (searched through every sub-folder)
   -> sample frames -> teacher masks (broiler / scale / feeder)     [Grounded SAM 2 or SAM 3]
   -> YOLO-seg dataset, split by video or by sub-folder
   -> train YOLOv8n-seg -> evaluate on held-out videos -> export NCNN / ONNX
   -> SAVE the model folder  ->  used by Notebook 3 (YOLO segmentation + fitted weight)
```

Weight estimation is **not** done here. Notebook 3 loads the model saved at the end of this notebook.

Set `DRY_RUN = True` to try it on generated nested folders of videos first.

## 0. Install (run once, then restart the kernel)

In [ ]:
INSTALL = False                  # True = install / update broilerseg from GitHub, then restart
BRANCH = "Fitting"               # branch
SUBDIR = "broilerseg 4"          # folder in the repo that holds pyproject.toml ("" = pick from a list)

if INSTALL:
    import shutil, subprocess, sys
    from pathlib import Path
    REPO = "https://github.com/Akintanoreofe/BroilerWeightEstimation.git"
    SRC = Path.home() / "broilerseg_src"
    shutil.rmtree(SRC, ignore_errors=True)
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(SRC)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)
    cands = sorted(p.parent.relative_to(SRC) for p in SRC.rglob("pyproject.toml")
                   if 'name = "broilerseg"' in p.read_text(errors="ignore"))
    print("broilerseg folders on the branch:", [str(c) for c in cands])
    pkg = SRC / SUBDIR if SUBDIR else SRC / cands[int(input(f"Pick 1-{len(cands)}: ")) - 1]
    if not (pkg / "pyproject.toml").is_file():
        raise FileNotFoundError(f"No pyproject.toml directly inside '{pkg.relative_to(SRC)}'")
    for cmd in ([sys.executable, "-m", "pip", "install", f"{pkg}[all]"],
                [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", str(pkg)]):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(r.stderr[-2500:])
    print("Installed from", pkg.relative_to(SRC), "- now restart the kernel and set INSTALL = False")

## Settings

In [ ]:
# ---- data ---------------------------------------------------------------------------
DRY_RUN = False
TEACHER = "grounded_sam2"        # "grounded_sam2" (public) or "sam3" (once Meta approves access)
FRAMES_PER_VIDEO = 15            # frames the teacher labels per video
SAMPLING = "uniform"             # "uniform", "random" or "motion" (busiest frames)
SKIP_START = 0                   # ignore the first N frames of each video
MIN_TEACHER_SCORE = 0.4          # drop low-confidence teacher masks from the training labels
# ---- dataset split -------------------------------------------------------------------
SPLIT_BY = "video"               # "video" or "folder" (whole sub-folders held out: harder test)
VAL_FRACTION = 0.15              # share of videos / folders for validation
TEST_FRACTION = 0.15             # share for a final test, never used during training
# ---- training ------------------------------------------------------------------------
RUN_TRAINING = True
BASE_MODEL = "yolov8n-seg.pt"    # nano = Raspberry-Pi size; "yolov8s-seg.pt" if accuracy is short
EPOCHS = 100
PATIENCE = 30                    # stop early when validation stops improving
IMGSZ = 640                      # use the size you will run on the Pi (320 / 480 are faster)
BATCH = 16
EXPORT_FORMATS = ("ncnn", "onnx")
# ---- where things go -----------------------------------------------------------------
MODEL_NAME = "broiler_yolov8n_seg"
MODELS_DIR = "~/broilerseg_models"                 # the saved model folder goes here
OUTPUT_DIR = "~/broilerseg_outputs/02_distillation"
SEED = 0
SAVE_FIGURES = True
# ---------------------------------------------------------------------------------------

In [ ]:
import json
import warnings
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import broilerseg
from broilerseg import viz
from broilerseg.config import CLASS_NAMES, PromptConfig
from broilerseg.distill import (benchmark, evaluate_student, export_for_raspberry_pi,
                                load_training_curves, path_size_mb, save_student,
                                teacher_student_table, train_student)
from broilerseg.io import motion_scores, read_frame, scan_videos
from broilerseg.labeling import auto_label_videos, load_result, plan_sampling
from broilerseg.notebook import ask_for_directory, environment_report
from broilerseg.yolo_dataset import build_yolo_dataset

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
viz.set_style()
OUT = Path(OUTPUT_DIR).expanduser().resolve(); (OUT / "figures").mkdir(parents=True, exist_ok=True)
MODELS = Path(MODELS_DIR).expanduser().resolve(); MODELS.mkdir(parents=True, exist_ok=True)
FIG = 0
def show(fig, name):
    global FIG
    FIG += 1
    if SAVE_FIGURES:
        viz.save(fig, OUT / "figures" / f"{FIG:02d}_{name}.png")
    plt.show()
try:
    import ultralytics  # noqa: F401
    HAVE_ULTRALYTICS = True
except ImportError:
    HAVE_ULTRALYTICS = False
print("broilerseg", broilerseg.__version__, "| ultralytics:", HAVE_ULTRALYTICS)
if broilerseg.__version__ < "0.4.0":
    print("This notebook needs broilerseg >= 0.4.0 - set INSTALL = True above, run it, restart.")
display(pd.Series(environment_report(), name="environment").to_frame())

## 1. The folder tree of videos

Paste the **top** folder. Every sub-folder is searched, however deep. Videos with the same file name in different folders (e.g. `ch01.mp4` in every day-folder) are told apart by their path: `day1__cam2__ch01`. To skip the prompt, set the `BROILERSEG_VIDEO_DIR` environment variable.

In [ ]:
if DRY_RUN:
    from broilerseg.synthetic import make_demo_data
    demo = make_demo_data(OUT / "demo_data", n_events=8, frames_per_event=30, nested=True)
    VIDEO_DIR = demo["videos"]
    print("DRY RUN: synthetic nested folders in", VIDEO_DIR)
else:
    VIDEO_DIR = ask_for_directory("Paste the TOP folder that contains the video sub-folders",
                                  env_var="BROILERSEG_VIDEO_DIR")
videos = scan_videos(VIDEO_DIR)
if videos.empty:
    raise FileNotFoundError(f"No videos found anywhere under {VIDEO_DIR}")
print(f"{len(videos)} videos in {videos.folder.nunique()} folders "
      f"({videos.duration_s.sum() / 60:.1f} min, {videos.size_mb.sum():.0f} MB)")
display(videos[["video_id", "folder", "name", "n_frames", "fps", "width", "height", "duration_s"]].head(20))
show(viz.plot_folder_summary(videos), "folders")
show(viz.plot_video_inventory(videos.assign(stem=videos.video_id)), "video_inventory")
firsts = videos.groupby("folder").head(1).head(15)
show(viz.plot_contact_sheet([read_frame(p, min(10, max(0, n - 1))) for p, n in zip(firsts.path, firsts.n_frames)],
                            list(firsts.folder), title="One frame from each folder"), "contact_sheet")

## 2. Which frames to label

In [ ]:
plan = plan_sampling(videos, FRAMES_PER_VIDEO, SAMPLING, SEED, SKIP_START)
plan.to_csv(OUT / "sampling_plan.csv", index=False)
print(f"{len(plan)} frames from {plan.video.nunique()} videos")
show(viz.plot_sampling_plan(plan), "sampling_plan")
v0 = videos.iloc[0]
mo = motion_scores(v0.path, stride=max(1, int(v0.n_frames) // 300))
show(viz.plot_motion_profile(mo, plan[plan.video == v0.path].frame_idx.tolist(),
                             title=f"Activity in {v0.video_id}"), "motion_profile")

## 3. Label the frames with the teacher

Frames, masks (`.npz`), overlays and `manifest.csv` are written to `labels/`. **Re-running resumes**: frames that already have masks are skipped, so a long run can be stopped and continued.

In [ ]:
LABEL_DIR = OUT / "labels"
prompts = PromptConfig()
if DRY_RUN:
    from broilerseg.synthetic import ColorSegmenter
    teacher = ColorSegmenter()
else:
    from broilerseg.segmentation import load_teacher
    teacher = load_teacher(TEACHER, prompts=prompts, max_side=1280)
    print(teacher.name, "on", teacher.device)
manifest = auto_label_videos(VIDEO_DIR, LABEL_DIR, teacher, plan=plan)
print(f"{len(manifest)} frames labelled -> {LABEL_DIR}")
show(viz.plot_label_statistics(manifest), "label_statistics")
def load_frame(stem):
    return cv2.cvtColor(cv2.imread(str(LABEL_DIR / "frames" / f"{stem}.jpg")), cv2.COLOR_BGR2RGB)
teacher_results = {st: load_result(LABEL_DIR / "masks" / f"{st}.npz") for st in manifest.stem}
rng = np.random.default_rng(SEED)
stems = rng.choice(manifest.stem, size=min(12, len(manifest)), replace=False)
show(viz.plot_gallery([(load_frame(st), teacher_results[st], st[-24:]) for st in stems], ncols=4,
                      title=f"{teacher.name} pseudo-labels"), "teacher_gallery")
show(viz.plot_segmentation_panels(load_frame(stems[0]), teacher_results[stems[0]], stems[0]), "teacher_panels")
suspect = manifest[(manifest.n_feeder != 1) | (manifest.n_scale != 1)]
print(f"{len(suspect)} frames with a missing / duplicated scale or feeder - check these:")
if len(suspect):
    show(viz.plot_gallery([(load_frame(s), teacher_results[s], s[-24:]) for s in suspect.stem[:8]],
                          ncols=4, show_scores=True, title="Check these pseudo-labels"), "teacher_suspect")

## 4. Build the YOLO-seg dataset

Masks become polygons. With `SPLIT_BY = "video"`, whole videos go to train / val / test. With `"folder"`, whole sub-folders do: a harder and more honest test if folders are different days or cameras.

In [ ]:
YOLO_DIR = OUT / "yolo_dataset"
ds = build_yolo_dataset(LABEL_DIR, YOLO_DIR, (1 - VAL_FRACTION - TEST_FRACTION, VAL_FRACTION, TEST_FRACTION),
                        seed=SEED, min_score=MIN_TEACHER_SCORE, split_by=SPLIT_BY)
print((YOLO_DIR / "data.yaml").read_text())
display(ds.groupby("split").agg(images=("stem", "size"), groups=("split_group", "nunique"),
                                **{f"{c}_polygons": (f"n_{c}", "sum") for c in CLASS_NAMES}))
show(viz.plot_split_summary(ds), "split_summary")
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
for ax, (_, r) in zip(axes, ds.sample(min(4, len(ds)), random_state=SEED).iterrows()):
    viz.plot_yolo_label_check(YOLO_DIR / "images" / r.split / f"{r.stem}.jpg",
                              YOLO_DIR / "labels" / r.split / f"{r.stem}.txt", ax=ax,
                              title=f"{r.split}: {r.stem[-20:]}")
fig.suptitle("Label files read back from disk (what YOLO trains on)", fontweight="bold"); fig.tight_layout()
show(fig, "yolo_label_check")

## 5. Train the student

In [ ]:
RUN_DIR = None
if RUN_TRAINING and HAVE_ULTRALYTICS:
    RUN_DIR = train_student(YOLO_DIR / "data.yaml", BASE_MODEL, EPOCHS, IMGSZ, BATCH,
                            project=OUT / "runs", name=MODEL_NAME, patience=PATIENCE, seed=SEED)
    print("Run folder:", RUN_DIR)
    curves = load_training_curves(RUN_DIR)
    show(viz.plot_training_curves(curves), "training_curves")
    pngs = [p for p in sorted(Path(RUN_DIR).glob("*.png")) if not p.name.startswith("train_batch")][:9]
    if pngs:
        show(viz.plot_contact_sheet([cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB) for p in pngs],
                                    [p.name for p in pngs], ncols=3, size=5.5,
                                    title="Ultralytics plots"), "ultralytics_plots")
else:
    print("Training skipped" + ("" if HAVE_ULTRALYTICS else " (ultralytics not installed)"))

## 6. Evaluate the student

1. **Ultralytics metrics** (mask and box mAP) on the validation and **test** videos, against the teacher's labels.
2. **Agreement with the teacher** on held-out frames: IoU per class, instance counts, and the frames where they disagree most.
3. **Speed and size** of the student vs the teacher, plus the Raspberry Pi exports.

In [ ]:
STUDENT_WEIGHTS = Path(RUN_DIR) / "weights" / "best.pt" if RUN_DIR else None
metrics_rows, per_class = [], None
if STUDENT_WEIGHTS and STUDENT_WEIGHTS.is_file():
    from broilerseg.segmentation import YoloSegmenter
    student = YoloSegmenter(STUDENT_WEIGHTS, imgsz=IMGSZ)
    for split in ("val", "test"):
        if (YOLO_DIR / "images" / split).is_dir() and any((YOLO_DIR / "images" / split).iterdir()):
            m, pc = evaluate_student(STUDENT_WEIGHTS, YOLO_DIR / "data.yaml", split, IMGSZ)
            metrics_rows.append(m)
            per_class = pc if split == "test" or per_class is None else per_class
    display(pd.DataFrame(metrics_rows).round(3))
    show(viz.plot_student_metrics(metrics_rows, per_class), "student_metrics")
else:
    from broilerseg.synthetic import ColorSegmenter
    student = ColorSegmenter(); student.name = "stand-in student (no trained model)"
    print("No trained model - a stand-in student is used so the plots below still appear.")

held = ds[ds.split == ("test" if (ds.split == "test").any() else "val")].stem.tolist() or ds.stem.tolist()
held_images = {s: load_frame(s) for s in held}
agree, student_results = teacher_student_table(held_images, teacher_results, student)
agree.to_csv(OUT / "teacher_student_agreement.csv", index=False)
display(agree.groupby("class_name").iou.describe().round(3))
show(viz.plot_iou_distribution(agree), "iou_distribution")
show(viz.plot_count_agreement(agree), "count_agreement")
per_img = agree.groupby("stem").iou.mean().sort_values()
for tag, ss in (("WORST", per_img.index[:3]), ("BEST", per_img.index[-2:])):
    for s in ss:
        show(viz.plot_teacher_student(held_images[s], teacher_results[s], student_results[s],
                                      f"{tag}: {s} (mean IoU {per_img[s]:.2f})"), f"ts_{tag.lower()}_{s[-12:]}")

In [ ]:
bench_imgs = list(held_images.values())[:8]
rows = [{**benchmark(teacher, bench_imgs[:3], warmup=1), "iou": 1.0,
         "size_mb": 0.0 if DRY_RUN else {"sam3": 3400, "grounded_sam2": 1800}[TEACHER]},
        {**benchmark(student, bench_imgs, warmup=2), "iou": float(agree.iou.mean()),
         "size_mb": path_size_mb(STUDENT_WEIGHTS) if STUDENT_WEIGHTS else 0.0}]
EXPORTS = {}
if STUDENT_WEIGHTS and STUDENT_WEIGHTS.is_file():
    from broilerseg.segmentation import YoloSegmenter
    EXPORTS = export_for_raspberry_pi(STUDENT_WEIGHTS, IMGSZ, EXPORT_FORMATS)
    for fmt, path in EXPORTS.items():
        seg = YoloSegmenter(path, imgsz=IMGSZ)
        tab, _ = teacher_student_table(dict(list(held_images.items())[:8]), teacher_results, seg)
        rows.append({**benchmark(seg, bench_imgs, warmup=2), "name": f"student {fmt}",
                     "iou": float(tab.iou.mean()), "size_mb": path_size_mb(path)})
speed = pd.DataFrame(rows)
display(speed.round(3))
show(viz.plot_speed_accuracy(rows), "speed_accuracy")

## 7. Save the model for Notebook 3

Everything needed to use the student goes into one folder: the weights, the Pi exports, the dataset definition, the training curves and a `model_card.json` with the evaluation results. **Notebook 3 asks for this folder.**

In [ ]:
card = {"imgsz": IMGSZ, "base_model": BASE_MODEL, "epochs": EPOCHS, "teacher": teacher.name,
        "teacher_key": TEACHER, "split_by": SPLIT_BY, "frames_per_video": FRAMES_PER_VIDEO,
        "videos": int(videos.shape[0]), "folders": int(videos.folder.nunique()),
        "images": {k: int(v) for k, v in ds.split.value_counts().items()},
        "metrics": metrics_rows, "mean_iou_vs_teacher": float(agree.iou.mean()),
        "iou_per_class": agree.groupby("class_name").iou.mean().round(4).to_dict(),
        "student_ms_per_frame_here": float(speed.ms_mean.iloc[1]),
        "video_root": str(VIDEO_DIR), "stand_in": STUDENT_WEIGHTS is None,
        "trained": datetime.now().isoformat(timespec="seconds")}
MODEL_DIR = save_student(RUN_DIR, MODELS, MODEL_NAME, EXPORTS, card)
print("Saved model folder:", MODEL_DIR)
for p in sorted(MODEL_DIR.iterdir()):
    print("  ", p.name, f"({path_size_mb(p):.1f} MB)")
print("\nIn Notebook 3 set  MODEL_DIR =", repr(str(MODEL_DIR)))

**Next:** open **Notebook 3**. It loads this saved model, segments with YOLO, fits the mass equation and estimates weights.

If the student disagrees with the teacher too often (section 6), label more frames (raise `FRAMES_PER_VIDEO`, add videos, fix the suspect frames in section 3), try `yolov8s-seg.pt`, or train at the image size you will deploy at.